In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from rapidfuzz import fuzz, process

import xgboost as xgb

PROJECT_ROOT = Path("..").resolve()

RESULTS_DIR = PROJECT_ROOT / "results"
OUTPUT_DIR = PROJECT_ROOT / "output"

CACHE_DIR = RESULTS_DIR / "inference_cache"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("CACHE_DIR:", CACHE_DIR)

PROJECT_ROOT: D:\V-Max_Hackerz
RESULTS_DIR: D:\V-Max_Hackerz\results
CACHE_DIR: D:\V-Max_Hackerz\results\inference_cache


In [2]:
import sys

SRC_DIR = (
    PROJECT_ROOT
    / "code"
    / "business_entity_resolution"
    / "src"
)

sys.path.insert(0, str(SRC_DIR))

from business_entity_resolution.preprocessing.normalization import (
    normalize_name,
    normalize_address
)

print("Normalization functions loaded.")

Normalization functions loaded.


In [3]:
s1_cache_file = CACHE_DIR / "test_s1_features.parquet"
candidate_cache_file = CACHE_DIR / "test_candidate_features.parquet"

print("S1 cache:", s1_cache_file)
print("Candidate cache:", candidate_cache_file)

print("S1 cache exists:", s1_cache_file.exists())
print("Candidate cache exists:", candidate_cache_file.exists())

S1 cache: D:\V-Max_Hackerz\results\inference_cache\test_s1_features.parquet
Candidate cache: D:\V-Max_Hackerz\results\inference_cache\test_candidate_features.parquet
S1 cache exists: True
Candidate cache exists: True


In [4]:
s1_cache = pd.read_parquet(s1_cache_file)
candidate_cache = pd.read_parquet(candidate_cache_file)

print("S1 cache shape:", s1_cache.shape)
print("Candidate cache shape:", candidate_cache.shape)

print("\nS1 columns:")
print(s1_cache.columns.tolist())

print("\nCandidate columns:")
print(candidate_cache.columns.tolist())

S1 cache shape: (1732544, 9)
Candidate cache shape: (9969589, 8)

S1 columns:
['entity_id', 'business_name', 'business_address', 'country', 'norm_name', 'norm_address', 'country_clean', 'name_missing', 'address_missing']

Candidate columns:
['business_name', 'business_address', 'country', 'norm_name', 'norm_address', 'country_clean', 'name_missing', 'address_missing']


In [15]:
# S1 cache
if "entity_id" in s1_cache.columns:
    s1_cache = s1_cache.set_index("entity_id", drop=False)
else:
    s1_cache.index.name = "entity_id"

# Candidate cache
if "entity_id" in candidate_cache.columns:
    candidate_cache = candidate_cache.set_index("entity_id", drop=False)
else:
    candidate_cache.index.name = "entity_id"

print("\nFinal S1 index:", s1_cache.index.name)
print("Final candidate index:", candidate_cache.index.name)

print("S1 shape:", s1_cache.shape)
print("Candidate shape:", candidate_cache.shape)

print("\nFirst candidate index values:")
print(candidate_cache.index[:5].tolist())


Final S1 index: entity_id
Final candidate index: entity_id
S1 shape: (1732544, 9)
Candidate shape: (9969589, 8)

First candidate index values:
['S2-192345572', 'S2-566025912', 'S2-158121477', 'S2-89663826', 'S2-884102769']


In [6]:
# Inspect indexes first
print("S1 index:", s1_cache.index.name)
print("Candidate index:", candidate_cache.index.name)

print("\nS1 columns:")
print(s1_cache.columns.tolist())

print("\nCandidate columns:")
print(candidate_cache.columns.tolist())

S1 index: entity_id
Candidate index: entity_id

S1 columns:
['entity_id', 'business_name', 'business_address', 'country', 'norm_name', 'norm_address', 'country_clean', 'name_missing', 'address_missing']

Candidate columns:
['business_name', 'business_address', 'country', 'norm_name', 'norm_address', 'country_clean', 'name_missing', 'address_missing']


In [17]:
from functools import lru_cache

@lru_cache(maxsize=2_000_000)
def cached_token_set(text):
    if text is None or pd.isna(text):
        return frozenset()

    text = str(text).strip()

    if not text:
        return frozenset()

    return frozenset(text.split())


print("Token cache functions ready.")

Token cache functions ready.


In [9]:
def jaccard_from_sets(a, b):
    if not a and not b:
        return 0.0

    union = a | b

    if not union:
        return 0.0

    return len(a & b) / len(union)


def safe_length_ratio(a, b):
    a = str(a) if not pd.isna(a) else ""
    b = str(b) if not pd.isna(b) else ""

    if not a and not b:
        return 1.0

    if not a or not b:
        return 0.0

    return min(len(a), len(b)) / max(len(a), len(b))

In [23]:
def build_chunk_features_fast(candidate_pairs):
    
    candidate_pairs = candidate_pairs[
        ["source1_entity_id", "candidate_entity_id"]
    ].copy()

    s1_ids = candidate_pairs["source1_entity_id"].to_numpy()
    candidate_ids = candidate_pairs["candidate_entity_id"].to_numpy()

    # --------------------------------------------------
    # Retrieve cached records
    # --------------------------------------------------

    s1_rows = s1_cache.reindex(s1_ids)
    candidate_rows = candidate_cache.reindex(candidate_ids)

    # --------------------------------------------------
    # Normalized strings
    # --------------------------------------------------

    s1_names = (
        s1_rows["norm_name"]
        .fillna("")
        .astype(str)
        .to_numpy()
    )

    cand_names = (
        candidate_rows["norm_name"]
        .fillna("")
        .astype(str)
        .to_numpy()
    )

    s1_addresses = (
        s1_rows["norm_address"]
        .fillna("")
        .astype(str)
        .to_numpy()
    )

    cand_addresses = (
        candidate_rows["norm_address"]
        .fillna("")
        .astype(str)
        .to_numpy()
    )

    # --------------------------------------------------
    # Country
    # --------------------------------------------------

    s1_country = (
        s1_rows["country_clean"]
        .fillna("")
        .astype(str)
        .to_numpy()
    )

    cand_country = (
        candidate_rows["country_clean"]
        .fillna("")
        .astype(str)
        .to_numpy()
    )

    # --------------------------------------------------
    # Missing flags
    # --------------------------------------------------

    name_missing = (
        np.logical_or(
            s1_names == "",
            cand_names == ""
        )
        .astype(np.float32)
    )

    address_missing = (
        np.logical_or(
            s1_addresses == "",
            cand_addresses == ""
        )
        .astype(np.float32)
    )

    # --------------------------------------------------
    # Exact matches
    # --------------------------------------------------

    name_exact = (
        (s1_names == cand_names) &
        (s1_names != "")
    ).astype(np.float32)

    address_exact = (
        (s1_addresses == cand_addresses) &
        (s1_addresses != "")
    ).astype(np.float32)

    country_match = (
        (s1_country == cand_country) &
        (s1_country != "")
    ).astype(np.float32)

    # --------------------------------------------------
    # RapidFuzz similarities
    # --------------------------------------------------

    name_char_similarity = (
        process.cpdist(
            s1_names,
            cand_names,
            scorer=fuzz.ratio,
            workers=-1
        ) / 100.0
    ).astype(np.float32)

    name_token_set_ratio = (
        process.cpdist(
            s1_names,
            cand_names,
            scorer=fuzz.token_set_ratio,
            workers=-1
        ) / 100.0
    ).astype(np.float32)

    name_token_sort_ratio = (
        process.cpdist(
            s1_names,
            cand_names,
            scorer=fuzz.token_sort_ratio,
            workers=-1
        ) / 100.0
    ).astype(np.float32)

    address_char_similarity = (
        process.cpdist(
            s1_addresses,
            cand_addresses,
            scorer=fuzz.ratio,
            workers=-1
        ) / 100.0
    ).astype(np.float32)

    address_token_set_ratio = (
        process.cpdist(
            s1_addresses,
            cand_addresses,
            scorer=fuzz.token_set_ratio,
            workers=-1
        ) / 100.0
    ).astype(np.float32)

    address_token_sort_ratio = (
        process.cpdist(
            s1_addresses,
            cand_addresses,
            scorer=fuzz.token_sort_ratio,
            workers=-1
        ) / 100.0
    ).astype(np.float32)

    # --------------------------------------------------
    # Token statistics
    # --------------------------------------------------

    name_token_jaccard = np.zeros(len(candidate_pairs), dtype=np.float32)
    address_token_jaccard = np.zeros(len(candidate_pairs), dtype=np.float32)

    name_shared_tokens = np.zeros(len(candidate_pairs), dtype=np.float32)
    address_shared_tokens = np.zeros(len(candidate_pairs), dtype=np.float32)

    name_length_ratio = np.zeros(len(candidate_pairs), dtype=np.float32)
    address_length_ratio = np.zeros(len(candidate_pairs), dtype=np.float32)

    # Token sets for this chunk only
    s1_name_values = s1_cache.reindex(s1_ids)["norm_name"].fillna("").tolist()
    candidate_name_values = candidate_cache.reindex(candidate_ids)["norm_name"].fillna("").tolist()
    
    s1_address_values = s1_cache.reindex(s1_ids)["norm_address"].fillna("").tolist()
    candidate_address_values = candidate_cache.reindex(candidate_ids)["norm_address"].fillna("").tolist()
    
    s1_name_chunk_tokens = [
        cached_token_set(x) for x in s1_name_values
    ]
    
    candidate_name_chunk_tokens = [
        cached_token_set(x) for x in candidate_name_values
    ]
    
    s1_address_chunk_tokens = [
        cached_token_set(x) for x in s1_address_values
    ]
    
    candidate_address_chunk_tokens = [
        cached_token_set(x) for x in candidate_address_values
    ]
    for i, (sid, cid) in enumerate(zip(s1_ids, candidate_ids)):

        s1_nt = s1_name_chunk_tokens[i]
        c_nt = candidate_name_chunk_tokens[i]
    
        s1_at = s1_address_chunk_tokens[i]
        c_at = candidate_address_chunk_tokens[i]

        name_token_jaccard[i] = jaccard_from_sets(
            s1_nt,
            c_nt
        )

        address_token_jaccard[i] = jaccard_from_sets(
            s1_at,
            c_at
        )

        name_shared_tokens[i] = len(s1_nt & c_nt)
        address_shared_tokens[i] = len(s1_at & c_at)

        name_length_ratio[i] = safe_length_ratio(
            s1_names[i],
            cand_names[i]
        )

        address_length_ratio[i] = safe_length_ratio(
            s1_addresses[i],
            cand_addresses[i]
        )

    # --------------------------------------------------
    # EXACT SAME 17 FEATURES AS TRAINING
    # --------------------------------------------------

    X = np.column_stack([
        name_exact,
        name_token_jaccard,
        name_char_similarity,
        name_token_set_ratio,
        name_token_sort_ratio,
        name_shared_tokens,
        name_length_ratio,

        address_exact,
        address_token_jaccard,
        address_char_similarity,
        address_token_set_ratio,
        address_token_sort_ratio,
        address_shared_tokens,
        address_length_ratio,

        country_match,
        name_missing,
        address_missing
    ]).astype(np.float32)

    return X, s1_ids, candidate_ids

In [11]:
xgb_json_path = RESULTS_DIR / "xgboost_entity_resolution.json"

print("XGB JSON:", xgb_json_path)
print("Exists:", xgb_json_path.exists())

xgb_model = xgb.XGBClassifier()
xgb_model.load_model(xgb_json_path)

print("XGB JSON model loaded successfully.")

XGB JSON: D:\V-Max_Hackerz\results\xgboost_entity_resolution.json
Exists: True
XGB JSON model loaded successfully.


In [12]:
LGB_DIR = RESULTS_DIR / "lgb_rerank_checkpoints"

lgb_files = sorted(
    LGB_DIR.glob("chunk_*.parquet")
)

print("LGB checkpoints:", len(lgb_files))
print("First:", lgb_files[0])

LGB checkpoints: 89
First: D:\V-Max_Hackerz\results\lgb_rerank_checkpoints\chunk_00000.parquet


In [13]:
test_df = pd.read_parquet(lgb_files[0])

print("Rows:", len(test_df))
print("Columns:")
print(test_df.columns.tolist())

print(test_df.head())

Rows: 48346
Columns:
['source1_entity_id', 'candidate_entity_id', 'lgb_probability']
  source1_entity_id candidate_entity_id  lgb_probability
0      S1-714132312        S3-625880872         0.998688
1      S1-714132312        S3-867809779         0.976911
2      S1-106407869        S2-705547832         0.960914
3      S1-106407869        S3-613056593         0.973223
4      S1-156285671        S2-172049742         0.999205


In [24]:
X_check, check_s1_ids, check_candidate_ids = (
    build_chunk_features_fast(
        test_df[
            [
                "source1_entity_id",
                "candidate_entity_id"
            ]
        ]
    )
)

print("Feature matrix:", X_check.shape)
print("Expected features: 17")
print("NaN count:", np.isnan(X_check).sum())
print("Inf count:", np.isinf(X_check).sum())

Feature matrix: (48346, 17)
Expected features: 17
NaN count: 0
Inf count: 0


In [25]:
xgb_check_prob = (
    xgb_model
    .predict_proba(X_check)[:, 1]
)

print("First 20 XGB probabilities:")
print(xgb_check_prob[:20])

print("\nMin:", xgb_check_prob.min())
print("Max:", xgb_check_prob.max())
print("Unique:", len(np.unique(xgb_check_prob)))

First 20 XGB probabilities:
[0.9934389  0.9853365  0.968316   0.96837217 0.994649   0.9695291
 0.92958796 0.8965896  0.9520551  0.9981066  0.98620474 0.99842787
 0.9363153  0.87922424 0.99555796 0.9969549  0.9708434  0.787685
 0.99790025 0.7864794 ]

Min: 0.4633054
Max: 0.99987686
Unique: 40960


In [26]:
# ============================================
# FULL HYBRID INFERENCE
# LGB + XGB probabilities
# ============================================

from pathlib import Path
import pandas as pd
import numpy as np
import gc
import time

HYBRID_DIR = RESULTS_DIR / "hybrid_xgb_prob_checkpoints"
HYBRID_DIR.mkdir(parents=True, exist_ok=True)

lgb_files = sorted(
    (RESULTS_DIR / "lgb_rerank_checkpoints").glob("chunk_*.parquet")
)

print("LGB checkpoints:", len(lgb_files))
print("Existing hybrid checkpoints:", len(list(HYBRID_DIR.glob("chunk_*.parquet"))))

for chunk_idx, lgb_file in enumerate(lgb_files):

    out_file = HYBRID_DIR / lgb_file.name

    # Resume support
    if out_file.exists():
        print(f"Skipping {chunk_idx}: already exists")
        continue

    start = time.time()

    print(f"\nProcessing chunk {chunk_idx}/{len(lgb_files)-1}")
    print("Input:", lgb_file.name)

    # Load LGB checkpoint
    chunk = pd.read_parquet(lgb_file)

    # Build the exact same 17 features
    X_chunk, chunk_s1_ids, chunk_candidate_ids = build_chunk_features_fast(
        chunk[["source1_entity_id", "candidate_entity_id"]]
    )

    # XGBoost probability
    xgb_prob = xgb_model.predict_proba(X_chunk)[:, 1]

    # Add XGB probability
    chunk["xgb_probability"] = xgb_prob.astype(np.float32)

    # Save
    chunk.to_parquet(
        out_file,
        index=False
    )

    print(
        f"Saved {out_file.name} | "
        f"rows={len(chunk):,} | "
        f"time={(time.time()-start)/60:.2f} min"
    )

    del chunk, X_chunk, xgb_prob
    gc.collect()

print("\nDONE")
print(
    "Hybrid checkpoints:",
    len(list(HYBRID_DIR.glob("chunk_*.parquet")))
)

LGB checkpoints: 89
Existing hybrid checkpoints: 89
Skipping 0: already exists
Skipping 1: already exists
Skipping 2: already exists
Skipping 3: already exists
Skipping 4: already exists
Skipping 5: already exists
Skipping 6: already exists
Skipping 7: already exists
Skipping 8: already exists
Skipping 9: already exists
Skipping 10: already exists
Skipping 11: already exists
Skipping 12: already exists
Skipping 13: already exists
Skipping 14: already exists
Skipping 15: already exists
Skipping 16: already exists
Skipping 17: already exists
Skipping 18: already exists
Skipping 19: already exists
Skipping 20: already exists
Skipping 21: already exists
Skipping 22: already exists
Skipping 23: already exists
Skipping 24: already exists
Skipping 25: already exists
Skipping 26: already exists
Skipping 27: already exists
Skipping 28: already exists
Skipping 29: already exists
Skipping 30: already exists
Skipping 31: already exists
Skipping 32: already exists
Skipping 33: already exists
Skippi

In [27]:
HYBRID_DIR = RESULTS_DIR / "hybrid_xgb_prob_checkpoints"

hybrid_files = sorted(HYBRID_DIR.glob("chunk_*.parquet"))

print("Hybrid checkpoints:", len(hybrid_files))

hybrid = pd.concat(
    [pd.read_parquet(f) for f in hybrid_files],
    ignore_index=True
)

print("Total rows:", len(hybrid))
print("Columns:", hybrid.columns.tolist())
print(hybrid.head())

Hybrid checkpoints: 89
Total rows: 4262476
Columns: ['source1_entity_id', 'candidate_entity_id', 'lgb_probability', 'xgb_probability', 'borderline', 'hybrid_keep']
  source1_entity_id candidate_entity_id  lgb_probability  xgb_probability  \
0      S1-714132312        S3-625880872         0.998688         0.000074   
1      S1-714132312        S3-867809779         0.976911         0.000074   
2      S1-106407869        S2-705547832         0.960914         0.000074   
3      S1-106407869        S3-613056593         0.973223         0.000074   
4      S1-156285671        S2-172049742         0.999205         0.000074   

   borderline  hybrid_keep  
0       False        False  
1       False        False  
2       False        False  
3       False        False  
4       False        False  


In [29]:
# ============================================
# HYBRID THRESHOLDS
# ============================================

XGB_THRESHOLD = 0.75
LGB_THRESHOLD = 0.75

print("XGB threshold:", XGB_THRESHOLD)
print("LGB threshold:", LGB_THRESHOLD)

XGB threshold: 0.75
LGB threshold: 0.75


In [33]:
# ============================================
# LOAD TEST S1
# ============================================

TEST_DIR = PROJECT_ROOT / "student_resource"/"dataset" / "test"

test_s1 = pd.read_csv(
    TEST_DIR / "test_source1.tsv",
    sep="\t",
    dtype=str
)

print("Test S1 shape:", test_s1.shape)
print("Test S1 columns:", test_s1.columns.tolist())
print("Unique S1 IDs:", test_s1["entity_id"].nunique())

Test S1 shape: (1732544, 4)
Test S1 columns: ['entity_id', 'business_name', 'business_address', 'country']
Unique S1 IDs: 1732544


In [34]:
# ============================================
# BUILD HYBRID MATCHING RESULTS
# ============================================

MATCHING_HYBRID = OUTPUT_DIR / "matching_results_hybrid.tsv"

# Use the hybrid rule
# Either model with probability >= 0.75
hybrid["hybrid_keep"] = (
    (hybrid["xgb_probability"] >= XGB_THRESHOLD) |
    (hybrid["lgb_probability"] >= LGB_THRESHOLD)
)

print("XGB threshold:", XGB_THRESHOLD)
print("LGB threshold:", LGB_THRESHOLD)
print("Predicted candidate matches:", hybrid["hybrid_keep"].sum())


# Keep only predicted matches
kept = hybrid.loc[
    hybrid["hybrid_keep"],
    ["source1_entity_id", "candidate_entity_id"]
].copy()


# Group candidates by S1
matched = (
    kept.groupby("source1_entity_id")["candidate_entity_id"]
    .apply(lambda x: ",".join(pd.unique(x)))
)


# Start from ALL test S1 IDs
matching_hybrid = pd.DataFrame({
    "source1_entity_id": test_s1["entity_id"].astype(str)
})

# Add matches; missing S1s become empty
matching_hybrid["matched_entity_ids"] = (
    matching_hybrid["source1_entity_id"]
    .map(matched)
    .fillna("")
)


# Safety checks
print("\nRows:", len(matching_hybrid))
print("Expected S1:", len(test_s1))
print("Duplicate S1:", matching_hybrid["source1_entity_id"].duplicated().sum())
print("Empty predictions:", (matching_hybrid["matched_entity_ids"] == "").sum())


# Save WITHOUT touching original matching_results.tsv
matching_hybrid.to_csv(
    MATCHING_HYBRID,
    sep="\t",
    index=False
)

print("\nSaved:")
print(MATCHING_HYBRID)

XGB threshold: 0.75
LGB threshold: 0.75
Predicted candidate matches: 4155598

Rows: 1732544
Expected S1: 1732544
Duplicate S1: 0
Empty predictions: 235732

Saved:
D:\V-Max_Hackerz\output\matching_results_hybrid.tsv
